In [2]:
!pip install xgboost


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [50]:
# GridSearchCV : 시도해볼 파라미터들을 표처럼 나열해 모든 조합을 하나씩 대입하여 여러번 검증해 평균 점수 낸다
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing
import ssl

In [51]:
# ssl 인증서 오류 방지 설정
ssl._create_default_https_context=ssl._create_unverified_context

In [52]:
housing=fetch_california_housing()
x_features=pd.DataFrame(housing.data, columns=housing.feature_names)
y_target=housing.target

In [53]:
# 학습/테스트데이터 분리
X_train, X_test, y_train, y_test=train_test_split(x_features,y_target,test_size=0.2, random_state=1)

In [54]:
# 피처, 라벨 스케일링
scalar=StandardScaler()
x_train_scaled=scalar.fit_transform(X_train)
y_test_scaled=scalar.transform(X_test)

In [55]:
# 모델 정의
model_configs=[
    {'name':'Ridge', 'estimator':Ridge(random_state=1), 'param':{'alpha':[1,10,100]}},
    {'name':'Lasso', 'estimator':Lasso(random_state=1), 'param':{'alpha':[0.001,0.01,0.1]}},
    {'name':'DecisionTree', 'estimator':DecisionTreeRegressor(random_state=1), 'param':{'max_depth':[5,10,15]}},
    {'name':'RandomForest', 'estimator':RandomForestRegressor(random_state=1), 'param':{'n_estimators':[100,200], 'max_depth':[10,20]}},
    {'name':'XGB', 'estimator':XGBRegressor(random_state=1), 'param':{'n_estimators':[100,200], 'max_depth':[5,8],'learning_rate':[0.05,0.1]}},
    {'name':'GradientBoosting', 'estimator':GradientBoostingRegressor(random_state=1), 'param':{'n_estimators':[100,200], 'max_depth':[5,8],'learning_rate':[0.05,0.1]}}
]

In [56]:
result={}

In [35]:
GridSearchCV?

Init signature:
GridSearchCV(
    estimator,
    param_grid,
    *,
    scoring=None,
    n_jobs=None,
    refit=True,
    cv=None,
    verbose=0,
    pre_dispatch='2*n_jobs',
    error_score=nan,
    return_train_score=False,
)
Docstring:     
Exhaustive search over specified parameter values for an estimator.

Important members are fit, predict.

GridSearchCV implements a "fit" and a "score" method.
It also implements "score_samples", "predict", "predict_proba",
"decision_function", "transform" and "inverse_transform" if they are
implemented in the estimator used.

The parameters of the estimator used to apply these methods are optimized
by cross-validated grid-search over a parameter grid.

Read more in the :ref:`User Guide <grid_search>`.

Parameters
----------
estimator : estimator object
    This is assumed to implement the scikit-learn estimator interface.
    Either estimator needs to provide a ``score`` function,
    or ``scoring`` must be passed.

param_grid : dict or list of

In [37]:
for config in model_configs:
    model_name=config['name']
    print(f"{model_name} loading...")
    # estimator : 어떤 알고리즘을 사용할지 결정
    # param_grid : 테스트 해 볼 파라미터 조합 리스트
    # cv : cross validation - 데이터를 5등분하여 5번 검증
    # scoring : 모델 평가하는 정수 규칙(오차가 작을수록 좋다 - GridSearchCV 는 점수가 높을수록 좋다고 판단 - 그래서 오차값에 - 붙여야함
    # refit : 모든 조합을 다 테스트한 후, 가장 점수가 높았던 파라미터로 전체 데이터를 다시 학습시킨다.
    grid_search=GridSearchCV(
        estimator=config['estimator'],
        param_grid=config['param'],
        cv=5,
        scoring='neg_mean_squared_error',
        refit=True,
        n_jobs=1,
    )

    grid_search.fit(x_train_scaled, y_train)

    # 모든 조합을 다 돌려본 후, 가장 성적이 좋은 파라미터 조합을 best_model에 저장
    best_model=grid_search.best_estimator_
    y_pred=best_model.predict(y_test_scaled)

    result[model_name]={
        'best_params':grid_search.best_params_,
        'best_neg_mse':grid_search.best_score_,
        'test_r2':r2_score(y_test, y_pred)
    }

results_df=pd.DataFrame(result).T.sort_values(by='test_r2', ascending=False)
print(results_df)

Ridge loading...
Lasso loading...
DecisionTree loading...
RandomForest loading...
XGB loading...
GradientBoosting loading...
                                                        best_params  \
XGB               {'learning_rate': 0.1, 'max_depth': 8, 'n_esti...   
GradientBoosting  {'learning_rate': 0.1, 'max_depth': 8, 'n_esti...   
RandomForest                 {'max_depth': 20, 'n_estimators': 200}   
DecisionTree                                      {'max_depth': 10}   
Ridge                                                 {'alpha': 10}   
Lasso                                              {'alpha': 0.001}   

                 best_neg_mse   test_r2  
XGB                  -0.21641  0.841718  
GradientBoosting    -0.221499  0.838804  
RandomForest        -0.260896  0.808045  
DecisionTree        -0.430651  0.703339  
Ridge               -0.528091  0.596678  
Lasso               -0.527927  0.596658  


In [48]:
results_df=pd.DataFrame(result)

In [47]:
RandomizedSearchCV?

Init signature:
RandomizedSearchCV(
    estimator,
    param_distributions,
    *,
    n_iter=10,
    scoring=None,
    n_jobs=None,
    refit=True,
    cv=None,
    verbose=0,
    pre_dispatch='2*n_jobs',
    random_state=None,
    error_score=nan,
    return_train_score=False,
)
Docstring:     
Randomized search on hyper parameters.

RandomizedSearchCV implements a "fit" and a "score" method.
It also implements "score_samples", "predict", "predict_proba",
"decision_function", "transform" and "inverse_transform" if they are
implemented in the estimator used.

The parameters of the estimator used to apply these methods are optimized
by cross-validated search over parameter settings.

In contrast to GridSearchCV, not all parameter values are tried out, but
rather a fixed number of parameter settings is sampled from the specified
distributions. The number of parameter settings that are tried is
given by n_iter.

If all parameters are presented as a list,
sampling without replacement is p

In [58]:
for config in model_configs:
    model_name=config['name']
    print(f"{model_name} loading...")
    # estimator : 어떤 알고리즘을 사용할지 결정
    # param_grid : 테스트 해 볼 파라미터 조합 리스트
    # cv : cross validation - 데이터를 5등분하여 5번 검증
    # scoring : 모델 평가하는 정수 규칙(오차가 작을수록 좋다 - GridSearchCV 는 점수가 높을수록 좋다고 판단 - 그래서 오차값에 - 붙여야함
    # refit : 모든 조합을 다 테스트한 후, 가장 점수가 높았던 파라미터로 전체 데이터를 다시 학습시킨다.
    grid_search=RandomizedSearchCV(
        estimator=config['estimator'],
        param_distributions=config['param'],
        cv=5,
        n_iter=10,
        scoring='neg_mean_squared_error',
        refit=True,
        n_jobs=1,
        random_state=1
    )

    grid_search.fit(x_train_scaled, y_train)

    # 모든 조합을 다 돌려본 후, 가장 성적이 좋은 파라미터 조합을 best_model에 저장
    best_model=grid_search.best_estimator_
    y_pred=best_model.predict(y_test_scaled)

    result[model_name]={
        'best_params':grid_search.best_params_,
        'best_neg_mse':grid_search.best_score_,
        'test_r2':r2_score(y_test, y_pred)
    }
results_df=pd.DataFrame(result)
print(results_df)
results_df=pd.DataFrame(result).T.sort_values(by='test_r2', ascending=False)
print(results_df)

Ridge loading...
Lasso loading...


C:\Users\hi\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\hi\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


DecisionTree loading...


C:\Users\hi\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomForest loading...


...

XGB loading...


C:\Users\hi\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


GradientBoosting loading...


C:\Users\hi\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


                                                        best_params  \
XGB               {'n_estimators': 200, 'max_depth': 8, 'learnin...   
GradientBoosting  {'n_estimators': 200, 'max_depth': 8, 'learnin...   
RandomForest                 {'n_estimators': 200, 'max_depth': 20}   
DecisionTree                                      {'max_depth': 10}   
Ridge                                                 {'alpha': 10}   
Lasso                                              {'alpha': 0.001}   

                 best_neg_mse   test_r2  
XGB                  -0.21641  0.841718  
GradientBoosting    -0.221499  0.838804  
RandomForest        -0.260896  0.808045  
DecisionTree        -0.430651  0.703339  
Ridge               -0.528091  0.596678  
Lasso               -0.527927  0.596658  


In [ ]:
# 파라미터를 넓게 잡고 RandomizedSearchCV를 돌린 후
# 찾은 범위에서 촘촘하게 후보를 정해 GridSearchCV로 정밀하게 튜닝하는 방식

# GridSearchCV -> 시간이 너무 오래 걸린다
# RandomizedSearchCV -> 100가지 중 10개만 랜덤으로 골라서 측정해줘